# 05 - Real-gap candidate reconstructions

Loads the precomputed real-gap candidate output table and shows both
candidate methods (TS-ICL satellite-proxy and the engineered hybrid
pipeline) side by side. Fully executable on the public data included in
this repository.

**Real gaps have no withheld ground truth.** The outputs shown here are
plausible candidate values, not validation evidence -- see
`docs/evidence_and_limitations.md` for how to interpret them responsibly.

In [1]:
import pandas as pd

real_gaps = pd.read_csv(
    "../results/chlorophyll/chlorophyll_real_gap_candidate_outputs.csv",
    parse_dates=["start_date", "end_date"],
)
real_gaps.head()

,gap_id,start_date,end_date,length_days,gap_class,seasons,extrapolation_beyond_validation,scenario_only_256day,tsicl_satellite_proxy_mean_pred_chl,tsicl_satellite_proxy_n_days,engineered_hybrid_mean_reconstructed_chl,engineered_hybrid_method,engineered_hybrid_n_days,note_artificial_validation,note_real_gap_caveat,note_256day_scenario
0,REAL_L010_20150701,2015-07-01,2015-07-14,14,medium,JJA,no,False,1.9258,14,1.0391,state_space_kalman,14,Artificial-gap validation (the canonical 681-g...,Real gaps have no withheld ground truth; these...,NaN
1,REAL_L010_20150804,2015-08-04,2015-08-18,15,medium,JJA,interpolation_within_range,False,4.1708,15,4.4743,state_space_kalman,15,Artificial-gap validation (the canonical 681-g...,Real gaps have no withheld ground truth; these...,NaN
2,REAL_L001_20150914,2015-09-14,2015-09-14,1,short,SON,no,False,12.5289,1,13.1607,gaussian_process,1,Artificial-gap validation (the canonical 681-g...,Real gaps have no withheld ground truth; these...,NaN
3,REAL_L001_20151019,2015-10-19,2015-10-19,1,short,SON,no,False,7.9674,1,7.7469,gaussian_process,1,Artificial-gap validation (the canonical 681-g...,Real gaps have no withheld ground truth; these...,NaN
4,REAL_L001_20151109,2015-11-09,2015-11-13,5,short,SON,interpolation_within_range,False,1.5203,5,1.5094,state_space_kalman,5,Artificial-gap validation (the canonical 681-g...,Real gaps have no withheld ground truth; these...,NaN


## Reproducing the inventory and assembly (not just reading the frozen file)

Everything above and below this cell reads the already-assembled, frozen
`chlorophyll_real_gap_candidate_outputs.csv`. The cells in this section
instead run the actual deterministic code
(`experiments.chlorophyll.real_gap_inventory`,
`assemble_real_gap_candidates`, `select_real_gap_reconstruction`) that
detects real gaps from the daily target and joins the per-method candidate
files. No TS-ICL call, no model fit: pure detection and joining over
already-frozen inputs, so this runs in well under a second.

In [2]:
import sys

sys.path.insert(0, "..")

from experiments.chlorophyll import real_gap_inventory as ri
from experiments.chlorophyll import select_real_gap_reconstruction as sr
from experiments.chlorophyll.assemble_real_gap_candidates import run_assembly

target_df = pd.read_csv(
    "../data/chlorophyll/chlorophyll_daily_target.csv", parse_dates=["date"]
).set_index("date").sort_index()

inventory = ri.detect_real_gaps(target_df)
print(f"Detected {len(inventory)} real gaps from the daily target's eligibility column alone "
      f"(no candidate file was read to find these boundaries).")
inventory.head(3)

Detected 128 real gaps from the daily target's eligibility column alone (no candidate file was read to find these boundaries).


,gap_id,start_date,end_date,length_days,gap_class,seasons,year_start,year_end,pre_edge_available,post_edge_available,interpolation_admissible,gap_edge_features_admissible,nearest_val_lengths,extrapolation_beyond_validation,notes
0,REAL_L010_20150701,2015-07-01,2015-07-14,14,medium,JJA,2015,2015,False,True,False,False,14,no,
1,REAL_L010_20150804,2015-08-04,2015-08-18,15,medium,JJA,2015,2015,True,True,True,True,14–30,interpolation_within_range,
2,REAL_L001_20150914,2015-09-14,2015-09-14,1,short,SON,2015,2015,True,True,True,True,1,no,


## Two different kinds of candidate: method-selected vs. independent

The two methods shown throughout this notebook are not symmetric:

- **`engineered_hybrid`** is a **method-selected candidate**: for each real
  gap, a deterministic rule ("Rule D") assigns exactly one component method
  by gap length -- GP (L1-3), a state-space Kalman smoother (L4-29), or a
  gap-edge residual model (L>=30) -- and that component's output becomes
  the reported value. `select_real_gap_reconstruction.route_real_gaps`
  below reproduces this routing decision exactly (not a new fit -- a
  lookup over which method *would have been* used, verified against the
  already-frozen output).
- **`tsicl_satellite_proxy`** is a single method applied uniformly to every
  gap -- no per-gap routing.

Neither is "the" selected final reconstruction across *both* methods --
per this project's benchmark-first framing, no single real-gap series is
presented as uniquely correct. They remain two independent, differently-
constructed candidates, shown side by side.

In [3]:
routed = sr.route_real_gaps(inventory)
excluded = routed[routed["assigned_method"].isna()]
print(f"Rule D assigns a method to {routed['assigned_method'].notna().sum()} of {len(routed)} real gaps.")
print(f"Excluded (no post-edge context available): {excluded['gap_id'].tolist()}")
routed[["gap_id", "length_days", "post_edge_available", "assigned_method"]].head(10)

Rule D assigns a method to 127 of 128 real gaps.
Excluded (no post-edge context available): ['REAL_OPEN_20260515']


,gap_id,length_days,post_edge_available,assigned_method
0,REAL_L010_20150701,14,True,state_space_kalman
1,REAL_L010_20150804,15,True,state_space_kalman
2,REAL_L001_20150914,1,True,gaussian_process
3,REAL_L001_20151019,1,True,gaussian_process
4,REAL_L001_20151109,5,True,state_space_kalman
5,REAL_L031_20151130,45,True,gap_edge_residual_model
6,REAL_L001_20160201,4,True,state_space_kalman
7,REAL_L001_20160616,6,True,state_space_kalman
8,REAL_L001_20160719,3,True,gaussian_process
9,REAL_L001_20160726,1,True,gaussian_process


## Running the deterministic assembly

Joins the two frozen per-method candidate files with the just-detected
inventory, validates the join (unique rows, exact gap-day coverage, dates
within the declared gap window, finite predictions, ordered quantiles), and
writes to `build/chlorophyll/real_gap_candidates/` -- never overwrites
`results/`.

In [4]:
from pathlib import Path

rc = run_assembly(Path("../build/chlorophyll/real_gap_candidates"))
print(f"\nExit code: {rc} (0 = validation passed)")

Assembled 128 gaps, 976 day-level rows -> ../build/chlorophyll/real_gap_candidates
{
  "inputs": {
    "engineered_hybrid_sha256": "fdff8a97002936e5f5137075fd1a3b7629bb1fe4b8042d258608abf52d2774f0",
    "tsicl_satellite_proxy_sha256": "e3f5fbe1c33cc4163e3fdeac3083912a41395614f145a532266027480739d634",
    "daily_target_sha256": "f9553abf559d224262061ea9002ca9fe33b81317891ced23d8c8d5b6cebc3b7a"
  },
  "n_real_gaps_inventoried": 128,
  "n_gaps_with_engineered_hybrid": 127,
  "n_gaps_with_tsicl_satellite_proxy": 128,
  "n_gaps_with_both": 127,
  "n_day_level_rows": 976,
  "validation_status": "PASSED",
  "timestamp_utc": "2026-08-05T17:19:26.684857+00:00"
}

Exit code: 0 (0 = validation passed)


## Gap-length distribution of real gaps

In [5]:
real_gaps["length_days"].describe()

count    128.000000
mean       7.625000
std       25.478415
min        1.000000
25%        1.000000
50%        1.000000
75%        2.000000
max      256.000000
Name: length_days, dtype: float64

In [6]:
comparison = real_gaps[[
    "gap_id", "start_date", "length_days",
    "tsicl_satellite_proxy_mean_pred_chl",
    "engineered_hybrid_mean_reconstructed_chl",
    "extrapolation_beyond_validation",
    "scenario_only_256day",
]]
comparison.head(15)

,gap_id,start_date,length_days,tsicl_satellite_proxy_mean_pred_chl,engineered_hybrid_mean_reconstructed_chl,extrapolation_beyond_validation,scenario_only_256day
0,REAL_L010_20150701,2015-07-01,14,1.9258,1.0391,no,False
1,REAL_L010_20150804,2015-08-04,15,4.1708,4.4743,interpolation_within_range,False
2,REAL_L001_20150914,2015-09-14,1,12.5289,13.1607,no,False
3,REAL_L001_20151019,2015-10-19,1,7.9674,7.7469,no,False
4,REAL_L001_20151109,2015-11-09,5,1.5203,1.5094,interpolation_within_range,False
5,REAL_L031_20151130,2015-11-30,45,3.0083,1.6629,yes,False
6,REAL_L001_20160201,2016-02-01,4,3.6192,3.5495,interpolation_within_range,False
7,REAL_L001_20160616,2016-06-16,6,0.9464,0.8845,interpolation_within_range,False
8,REAL_L001_20160719,2016-07-19,3,2.2437,2.9360,no,False
9,REAL_L001_20160726,2016-07-26,1,2.4046,2.4491,no,False


## Where the two methods diverge

Large divergence between the two candidate methods on a given real gap is
a useful (informal) red flag -- it suggests at least one of them is
extrapolating into an unfamiliar regime for that gap, even though we can't
say which one (no ground truth exists to check).

In [7]:
comparison = comparison.copy()
comparison["abs_divergence"] = (
    comparison["tsicl_satellite_proxy_mean_pred_chl"]
    - comparison["engineered_hybrid_mean_reconstructed_chl"]
).abs()
comparison.sort_values("abs_divergence", ascending=False).head(10)

,gap_id,start_date,length_days,tsicl_satellite_proxy_mean_pred_chl,engineered_hybrid_mean_reconstructed_chl,extrapolation_beyond_validation,scenario_only_256day,abs_divergence
99,REAL_L001_20190226,2019-02-26,3,10.7613,13.3594,no,False,2.5981
52,REAL_L001_20170823,2017-08-23,6,8.9334,11.4508,interpolation_within_range,False,2.5174
58,REAL_L001_20171118,2017-11-18,1,16.8298,18.8083,no,False,1.9785
56,REAL_L001_20171018,2017-10-18,1,13.5364,15.5037,no,False,1.9673
116,REAL_L010_20231116,2023-11-16,22,2.9725,1.2210,interpolation_within_range,False,1.7515
32,REAL_L001_20161225,2016-12-25,1,5.5481,7.1266,no,False,1.5785
105,REAL_L091_20200211,2020-02-11,256,3.4719,4.9920,yes,True,1.5201
60,REAL_L001_20171202,2017-12-02,1,4.3595,5.7805,no,False,1.4210
120,REAL_L031_20240325,2024-03-25,43,3.5207,4.8909,yes,False,1.3702
14,REAL_L001_20160816,2016-08-16,1,9.0073,10.3683,no,False,1.3610


## The 256-day scenario gap

In [8]:
scenario_rows = real_gaps[real_gaps["scenario_only_256day"] == True]
scenario_rows[["gap_id", "start_date", "end_date", "length_days", "note_256day_scenario"]]

,gap_id,start_date,end_date,length_days,note_256day_scenario
105,REAL_L091_20200211,2020-02-11,2020-10-23,256,"Scenario-only output, far outside the validate..."


## Reminder

- Artificial-gap validation (notebooks 02-04) supports ranking methods --
  gaps there have a withheld, known true value. Every real gap shown in
  this notebook does not; every value above is a **candidate**, never an
  observation.
- `engineered_hybrid` is a method-*selected* candidate (Rule D, length-
  routed); `tsicl_satellite_proxy` is a single method applied uniformly.
  Neither is promoted as the one correct reconstruction across both.
- The 256-day gap is far outside the validated gap-length envelope
  (maximum validated length: 60 days) and should be treated as illustrative
  only -- flagged `scenario_only_256day` throughout.
- `experiments/chlorophyll/real_gap_contract.py` is the single source of
  truth for which published artifact is which kind of evidence -- consult
  it (`REAL_GAP_ARTIFACTS`) before citing any real-gap number elsewhere.